# Consumer Finance Complaint Intelligence — Executive EDA

Dieses Notebook bildet die **narrative Analyseebene** der Consumer Finance Complaint Intelligence Platform.

Es fasst die wichtigsten Ergebnisse der reproduzierbaren Python-Pipeline für den Zeitraum **2022 bis 2025** zusammen.

Die eigentliche Produktionslogik befindet sich bewusst nicht im Notebook, sondern in `src/`, damit Datenaufbereitung, Qualitätskontrollen, Kennzahlen, Sensitivitätsanalyse und Visualisierung automatisiert getestet werden können.

## Analyseumfang

- 10.269.540 veröffentlichte CFPB-Beschwerden
- 5.601 Unternehmen
- 11 harmonisierte Produktgruppen
- vollständige Kalenderjahre 2022–2025
- Datenbereinigung und Taxonomie-Harmonisierung
- Zeitreihen- und Korrelationsanalyse
- Segment-Sensitivitätsanalyse
- 24 automatisierte Tests

> Das Notebook verwendet die bereits erzeugten kleinen Reportdateien unter `reports/`. Dadurch kann die Executive EDA betrachtet und ausgeführt werden, ohne den mehrere Gigabyte großen Rohdatensatz erneut einzulesen.


## Interpretationsrahmen

Die CFPB Consumer Complaint Database ist **keine repräsentative Stichprobe aller Kundenerfahrungen**.

Hohe Beschwerdezahlen sind deshalb ein **Priorisierungs- und Untersuchungssignal**, aber kein unmittelbarer Beweis für schlechte Unternehmens- oder Produktqualität.

Für faire Unternehmensvergleiche wären zusätzliche Bezugsgrößen erforderlich, beispielsweise Kundenanzahl, Konten, Verträge, Marktanteil oder Produkt-Exposure.

Außerdem gilt für alle statistischen Ergebnisse:

**Korrelation ist keine Kausalität.**


In [ ]:
from pathlib import Path
import json

import pandas as pd


cwd = Path.cwd()

if (cwd / "reports").exists():
    project_root = cwd
elif (cwd.parent / "reports").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError(
        "Das Projektverzeichnis mit dem Ordner 'reports' wurde nicht gefunden."
    )

reports_dir = project_root / "reports"
figures_dir = reports_dir / "figures"

print(f"Projektverzeichnis: {project_root}")
print(f"Reports: {reports_dir}")


# 1. Executive KPIs

Die Analyse verarbeitet mehr als zehn Millionen veröffentlichte Beschwerden und konzentriert sich auf vollständige Kalenderjahre.

Die wichtigsten Portfolio-Kennzahlen werden direkt aus dem automatisch erzeugten KPI-Report geladen.


In [ ]:
kpis = pd.read_csv(reports_dir / "kpi_summary.csv")

metric_labels = {
    "complaints": "Beschwerden",
    "unique_companies": "Unternehmen",
    "unique_products": "Harmonisierte Produkte",
    "timely_response_rate": "Timely-Response-Rate",
    "narrative_share": "Narrative-Anteil",
    "median_days_to_company": "Median Tage bis Unternehmen",
}

kpis["Kennzahl"] = kpis["metric"].map(metric_labels).fillna(kpis["metric"])
kpis[["Kennzahl", "value"]].rename(columns={"value": "Wert"})


# 2. Zeitliche Entwicklung

Das veröffentlichte Beschwerdevolumen steigt von **800.245 Beschwerden im Jahr 2022** auf **5.442.977 Beschwerden im Jahr 2025**.

Das entspricht einer Veränderung von rund **+580,2 %**.

Der rollierende 12-Monats-Durchschnitt wird erst dargestellt, wenn tatsächlich zwölf vollständige Monatsbeobachtungen vorliegen.

![Monatliche Beschwerdeentwicklung](../reports/figures/01_monthly_complaints.png)


In [ ]:
yearly = pd.read_csv(reports_dir / "yearly_summary.csv")

yearly_display = yearly.copy()

yearly_display["timely_response_rate"] = (
    yearly_display["timely_response_rate"] * 100
).round(2)

yearly_display["narrative_share"] = (
    yearly_display["narrative_share"] * 100
).round(2)

yearly_display["yoy_growth"] = (
    yearly_display["yoy_growth"] * 100
).round(2)

yearly_display.rename(
    columns={
        "year": "Jahr",
        "complaints": "Beschwerden",
        "timely_response_rate": "Timely Response (%)",
        "narrative_share": "Narrative-Anteil (%)",
        "unique_companies": "Unternehmen",
        "yoy_growth": "YoY-Wachstum (%)",
    }
)


# 3. Produktkonzentration

Die Analyse zeigt eine außergewöhnlich starke Konzentration auf:

**Credit reporting or other personal consumer reports**

Dieses Segment umfasst im gesamten Analysezeitraum **8.824.129 Beschwerden** beziehungsweise rund **85,9 %** des Portfolios.

![Produktmix](../reports/figures/02_product_mix.png)

Die lineare Skalierung bleibt bewusst erhalten. Dadurch wird die tatsächliche Dominanz des Credit-Reporting-Segments nicht visuell relativiert.


In [ ]:
products = pd.read_csv(reports_dir / "product_summary.csv")

product_display = products.head(10).copy()

product_display["complaint_share"] = (
    product_display["complaint_share"] * 100
).round(2)

product_display["timely_response_rate"] = (
    product_display["timely_response_rate"] * 100
).round(2)

product_display[
    [
        "product",
        "complaints",
        "complaint_share",
        "timely_response_rate",
    ]
].rename(
    columns={
        "product": "Produkt",
        "complaints": "Beschwerden",
        "complaint_share": "Portfolio-Anteil (%)",
        "timely_response_rate": "Timely Response (%)",
    }
)


# 4. Timely Response

Die veröffentlichte Timely-Response-Rate des Gesamtportfolios liegt bei rund **99,6 %**.

Zwischen Produktgruppen bestehen jedoch relevante Unterschiede.

Besonders auffällig ist **Student Loan** mit rund **84,8 %**.

![Timely-Response-Rate](../reports/figures/03_timely_response_by_product.png)

Diese Werte sind explorative Screening-Signale. Ohne geeignete Exposure-Nenner stellen sie keine faire Bewertung einzelner Unternehmen oder Produktgruppen dar.


# 5. Produkt-Issue-Hotspots

Der größte harmonisierte Produkt-Issue-Hotspot ist:

**Credit Reporting → Incorrect information on your report**

mit **4.653.591 Beschwerden**.

Gemeinsam mit zwei weiteren Credit-Reporting-Issues konzentriert sich ein sehr großer Teil des Gesamtportfolios auf wenige Beschwerdegründe.

![Produkt-Issue-Hotspots](../reports/figures/04_issue_hotspots.png)


In [ ]:
hotspots = pd.read_csv(reports_dir / "issue_hotspots.csv")

hotspots[
    [
        "product",
        "issue",
        "complaints",
        "timely_response_rate",
        "growth_first_to_last",
    ]
].head(10).rename(
    columns={
        "product": "Produkt",
        "issue": "Issue",
        "complaints": "Beschwerden",
        "timely_response_rate": "Timely-Response-Rate",
        "growth_first_to_last": "Wachstum erstes→letztes Jahr",
    }
)


# 6. Produktentwicklung nach Jahr

Die Produkt-Jahres-Heatmap macht sichtbar, wie stark sich die Produktstruktur im Zeitraum verändert.

![Produkt-Jahres-Heatmap](../reports/figures/05_product_year_heatmap.png)

Die **Farbskala ist logarithmisch**, damit kleinere Produktgruppen trotz der extremen Credit-Reporting-Dominanz sichtbar bleiben.

Die Zahlen in den einzelnen Zellen bleiben absolute Beschwerdewerte.


# 7. Sensitivitätsanalyse: Credit Reporting

Die stärkste methodische Erkenntnis der EDA entsteht durch die Kontrolle des dominanten Credit-Reporting-Segments.

Der Anteil dieses Segments steigt von:

- **75,3 % im Jahr 2022**
- auf **88,4 % im Jahr 2025**

Auch ohne Credit Reporting steigt das veröffentlichte Beschwerdevolumen deutlich:

- 2022: **197.553**
- 2025: **632.673**
- Veränderung: **+220,3 %**

Der deskriptive lineare Monatstrend verändert sich jedoch erheblich:

- Gesamtportfolio: rund **+10.511 Beschwerden pro Monat**
- ohne Credit Reporting: rund **+925 Beschwerden pro Monat**

Rechnerisch entfallen damit rund **91,2 % der Steigung des linearen Gesamttrends** auf den Credit-Reporting-Anteil der monatlichen Beschwerdereihe.

![Credit-Reporting-Sensitivitätsanalyse](../reports/figures/06_credit_reporting_sensitivity.png)


In [ ]:
sensitivity_yearly = pd.read_csv(
    reports_dir / "segment_sensitivity_yearly.csv"
)

sensitivity_display = sensitivity_yearly.copy()

sensitivity_display["focus_product_share"] = (
    sensitivity_display["focus_product_share"] * 100
).round(2)

sensitivity_display.rename(
    columns={
        "year": "Jahr",
        "total_complaints": "Gesamtportfolio",
        "focus_product_complaints": "Credit Reporting",
        "without_focus_product": "Ohne Credit Reporting",
        "focus_product_share": "Credit-Reporting-Anteil (%)",
    }
)


In [ ]:
with (reports_dir / "segment_sensitivity_summary.json").open(
    encoding="utf-8"
) as file:
    sensitivity = json.load(file)

correlations = sensitivity["correlations"]

pd.DataFrame(
    {
        "Zusammenhang": [
            "Beschwerden ↔ Timely Response",
            "Beschwerden ↔ Narrative-Anteil",
            "Timely Response ↔ Narrative-Anteil",
        ],
        "Pearson Gesamt": [
            correlations[
                "complaints_vs_timely_response_rate"
            ]["all"]["pearson_r"],
            correlations[
                "complaints_vs_narrative_share"
            ]["all"]["pearson_r"],
            correlations[
                "timely_response_rate_vs_narrative_share"
            ]["all"]["pearson_r"],
        ],
        "Pearson ohne Credit Reporting": [
            correlations[
                "complaints_vs_timely_response_rate"
            ]["without_focus_product"]["pearson_r"],
            correlations[
                "complaints_vs_narrative_share"
            ]["without_focus_product"]["pearson_r"],
            correlations[
                "timely_response_rate_vs_narrative_share"
            ]["without_focus_product"]["pearson_r"],
        ],
    }
).round(3)


## Interpretation des Kompositionseffekts

Die Sensitivitätsanalyse zeigt, dass aggregierte Portfolio-Kennzahlen erheblich vom Produktmix geprägt werden.

Besonders deutlich wird dies bei den monatlichen Korrelationen:

- Beschwerden ↔ Narrative-Anteil: **Pearson r = −0,891** im Gesamtportfolio, aber nur **−0,227** ohne Credit Reporting.
- Beschwerden ↔ Timely Response: **Pearson r = +0,343** im Gesamtportfolio und **−0,297** ohne Credit Reporting.

Das Projekt bezeichnet diesen Befund bewusst als **Segment- beziehungsweise Kompositionseffekt**.

Es wird **nicht vorschnell ein Simpson-Paradox behauptet**, da dafür eine systematischere Prüfung der Beziehungen innerhalb sämtlicher relevanter Gruppen erforderlich wäre.

Für Management und Consulting folgt daraus:

> Aggregierte Portfolio-KPIs sollten durch segmentierte Analysen ergänzt werden, bevor Ursachen, operative Risiken oder Customer Outcomes abgeleitet werden.


# 8. Datenbereinigung, Taxonomie und Quality Gates

Die analytische Aussagekraft hängt nicht nur von Kennzahlen ab, sondern bereits von der Qualität der Datenbasis.

Die Pipeline trennt deshalb ausdrücklich:

- ursprüngliche Source-Werte,
- Missing-Value-Behandlung,
- historische Taxonomie-Harmonisierung.

Im Rohdatensatz fehlen bei **6 Beschwerden** veröffentlichte Issue-Werte.

Diese Beschwerden werden nicht entfernt. Der Source-Wert `issue` bleibt leer, während analytisch `harmonized_issue = "Issue not provided"` verwendet wird.

Zusätzlich sind rund **13,0 %** der Beschwerden von einer Produktharmonisierung betroffen.

Der aktuelle Analysebestand besteht sämtliche definierten strukturellen Quality Gates:

**`passed = true`**


In [ ]:
with (reports_dir / "data_quality_report.json").open(
    encoding="utf-8"
) as file:
    quality = json.load(file)

with (reports_dir / "data_cleaning_summary.json").open(
    encoding="utf-8"
) as file:
    cleaning = json.load(file)

with (reports_dir / "taxonomy_harmonization_summary.json").open(
    encoding="utf-8"
) as file:
    taxonomy = json.load(file)

pd.DataFrame(
    [
        {
            "Prüfung": "Quality Gate",
            "Wert": quality["passed"],
        },
        {
            "Prüfung": "Doppelte Complaint IDs",
            "Wert": quality["duplicate_complaint_ids"],
        },
        {
            "Prüfung": "Fehlende Source-Issues",
            "Wert": cleaning["source_missing_issue_rows"],
        },
        {
            "Prüfung": "Unbehandelte Missing-Issues",
            "Wert": cleaning["missing_issue_rows_unhandled"],
        },
        {
            "Prüfung": "Produktharmonisierungen",
            "Wert": taxonomy["product_rows_changed"],
        },
    ]
)


# 9. Business- und Consulting-Relevanz

Die Analyse lässt sich auf typische Fragestellungen in Banken und Finanzdienstleistungsunternehmen übertragen:

- Welche Produkte oder Prozesse erzeugen besonders viele Beschwerden?
- Welche Issue-Cluster sollten für Root-Cause-Analysen priorisiert werden?
- Welche Bereiche zeigen ungewöhnliche Veränderungen?
- Wie unterscheiden sich Response-Kennzahlen zwischen Segmenten?
- Welche Portfolio-Aggregate werden durch einzelne Geschäftsbereiche dominiert?
- Welche zusätzlichen internen Daten wären für eine belastbare Bewertung erforderlich?

Für Versicherungen ist nicht der konkrete CFPB-Datensatz, sondern insbesondere die Methodik übertragbar:

- Beschwerde- und Schadenklassifikation,
- Taxonomie-Harmonisierung,
- Bearbeitungszeiten,
- Issue-Hotspots,
- Segmentkontrollen,
- Datenqualitäts-Gates,
- Management-Reporting.

Für Technical AI Consulting demonstriert das Projekt zusätzlich den vollständigen Weg von einer fachlichen Fragestellung über eine reproduzierbare Datenpipeline bis zu einer methodisch kontrollierten Management-Interpretation.


# 10. Fazit

Die EDA zeigt drei zentrale Punkte:

1. Das veröffentlichte CFPB-Beschwerdevolumen wächst im Zeitraum 2022–2025 sehr stark.
2. Credit Reporting dominiert sowohl das Gesamtvolumen als auch die beobachtete Trendsteigerung.
3. Aggregierte statistische Beziehungen verändern sich deutlich, sobald dieses dominante Segment kontrolliert wird.

Der wichtigste methodische Schluss ist deshalb nicht allein eine einzelne Kennzahl:

> **Datenqualität, Taxonomiestabilität und Segmentstruktur müssen vor der Interpretation von Portfolio-KPIs gemeinsam berücksichtigt werden.**

Das Projekt bleibt bewusst eine **Exploratory Data Analysis**.

Prädiktives Machine Learning, NLP auf Consumer Narratives und produktive AI-Systeme sind mögliche spätere Erweiterungen, aber nicht Bestandteil dieses Modul-1-Abschlussprojekts.
